# v65 生PF提出 — **struct20 + livefield(struct場を test 既知帯込みでその場構築)**
- パラメータは struct20 best(LB 7.881)と同一。**変更点は struct 場のみ**: train場(npz)に、
  **全 test 坑井の既知帯(TVT_input prefix)から抽出した dip 観測点を追加**して場を構築する。
- 抽出は train 場ビルダー(struct_prior_v38)と同一手順: 水平区間の (R0−U) をランプ基底で最小二乗 → (中点XY, c/proj)。
  自坑井の prefix も自分の場に入る(TVT_input は与えられた入力=合法)。κ/σ較正は train 由来を流用。
- 判定: LB 7.881(struct20)との差分が「test既知帯で場を増強する価値」。
- dataset に struct_field_v38.npz 必須(rogii-v64-artifacts 等)。

In [ ]:
# ===== 設定(ここだけ編集) =====
import os, json, time, glob as _glob, math, warnings, pickle
import numpy as np, pandas as pd
from pathlib import Path
import torch
warnings.filterwarnings("ignore"); np.seterr(all="ignore")

# --- PFパラメータセット(ハードコード。tune_pf_v65 の params をそのまま貼る) ---
PARAM_SETS = [
    # struct20 = v38_anchor_mult20_current best(mult20, n_particles600)。258坑井別seed 7.65-7.72 実測
    {"use_phys": False, "use_anchor": True, "anchor_ramp_md": 0.0, "lk_floor": 0.0, "gr_debias": 0, "rate_target": "zero", "grid_step": 0.2, "tw_source": "tw", "self_mix_w": 1.0, "nbr_mix_w": 1.0, "n_particles": 600, "likelihood_scale": 29.056396339290437, "init_pos_std": 1.6168048868046325, "init_rate_std": 0.014412715870661067, "mom": 0.9995682504305502, "vn": 0.000873883338794689, "pn": 0.00606209538025288, "rp": 0.14025726060525087, "rr": 0.00398567931602027, "resamp": 0.18645022443117387, "gr_sig_min": 10.63834771185878, "gr_sig_max": 43.95858017409127, "gr_sig_def": 86.87561008087175, "gr_sig_mult": 2.7681905590609426, "tvt_clip_margin": 86.30698497112195, "rate_clip": 0.0, "gr_power": 1.85175446220882, "hgr_smooth_r": 2, "tw_gr_smooth_r": 0, "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 2.3072090423765363, "anchor_mult": 20.0},
]
N_SEEDS = 32          # seedアンサンブル(尤度加重平均)
SEED    = 777
AVG     = "mean"      # 複数セットの合成: "mean" | "median"
PP      = None        # 例 dict(tau=40.0, alpha=1.03): d=(pf-lk)*(1-exp(-mds/tau))*alpha。None=生値
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE   = torch.float32
CHUNK   = 60

def _detect_data():
    for _p in ["/kaggle/input/rogii-wellbore-geology-prediction",
               "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
               r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction"]:
        if os.path.exists(_p): return Path(_p)
    _h = _glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    return Path(os.path.dirname(_h[0]))
DATA = _detect_data(); TESTD = DATA / "test"
print("DATA:", DATA, "| device:", DEVICE, "| sets:", len(PARAM_SETS))

In [ ]:
# ===== PFエンジン(pf_engine_v65.py から抽出した純計算部) =====
def _bin_pairs(tvt, gr, gmin, step, G):
    """(TVT,GR)点群 → 格子bin中央値(grid_step共通)。無データbin=NaN。"""
    out = np.full(G, np.nan)
    idx = np.round((np.asarray(tvt, float) - gmin) / step).astype(int)
    ok = (idx >= 0) & (idx < G) & np.isfinite(gr) & np.isfinite(tvt)
    if ok.sum():
        srt = pd.Series(np.asarray(gr, float)[ok]).groupby(idx[ok]).median()
        out[srt.index.to_numpy()] = srt.to_numpy()
    return out

def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2 * r + 1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b + step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def make_inputs(c, P):
    tw_gr_pf = _smooth(c["tw_gr"], c["tw_fb"], P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(c["tw_tvt"], tw_gr_pf, step=float(P.get("grid_step", 0.2)))
    gr_sm = _smooth(c["gr_filled"], c["tw_fb"], P["hgr_smooth_r"])
    bias = 0.0
    if len(c["kpos"]) < 20:
        gs = P["gr_sig_def"]
    else:
        resid = gr_sm[c["kpos"]] - np.interp(c["kn_tvtin"], c["tw_tvt"], tw_gr_pf)
        gs = float(np.nanstd(resid)); gs = P["gr_sig_def"] if (not np.isfinite(gs) or gs <= 0) else gs
        if int(P.get("gr_debias", 0)):
            b_ = float(np.nanmedian(resid))
            if np.isfinite(b_): bias = b_       # 井戸レベルGRオフセット(工具較正差)を尤度前に補正
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    if bias != 0.0:
        gr_sm = gr_sm - bias
    # ---- self/neighbor 疑似typewell混合(σ較正後。grid_step は全格子共通) ----
    src = str(P.get("tw_source", "tw"))
    if src != "tw":
        gg = gg.copy()
        if "nbr" in src and c.get("nbr_pairs") is not None:
            nt, ng = c["nbr_pairs"]
            ngs = _smooth(ng, float(np.nanmean(ng)), P["hgr_smooth_r"]).astype(np.float64)
            gh = _bin_pairs(nt, ngs, gmin, gst, len(gg))
            have = np.isfinite(gh); w = float(P.get("nbr_mix_w", 1.0))
            gg[have] = w * gh[have] + (1 - w) * gg[have]
        if "self" in src:
            gh = _bin_pairs(c["kn_tvtin"], gr_sm[c["kpos"]].astype(np.float64), gmin, gst, len(gg))
            have = np.isfinite(gh); w = float(P.get("self_mix_w", 1.0))
            gg[have] = w * gh[have] + (1 - w) * gg[have]
    return dict(md=c["md"], z=c["z"], gr=gr_sm[c["ev_pos"]].astype(np.float64),
                gg=gg, gmin=gmin, gst=gst, gs=gs, ls=c["ls"], ir=c["ir"], N=int(P["n_particles"]),
                anc=c["anc"], ancs=c["ancs"])

def _pf_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, anc, ancs, P, N, device, gen, dtype=DTYPE):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    USE_A = bool(P.get("use_anchor", True)); AMUL = float(P.get("anchor_mult", 20.0))
    RAMP = float(P.get("anchor_ramp_md", 0.0))          # >0: 錨をmdsでランプイン(近錨域の引きずり防止)
    LKF = float(P.get("lk_floor", 0.0))                  # >0: 尤度の一様混合床(外れGRロバスト化)
    RT_IR = (str(P.get("rate_target", "zero")) == "ir")  # momentum減衰先: 0 or 初期dip
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS * rn(); rate = ir[:, None] + IR * rn()
    w = torch.full((B, N), 1.0 / N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype) - 1) * step + CLIP
    glast = torch.gather(grid, 1, (glen - 1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur - pm).clamp_min(1.0)
        rate_n = ALPHA * rate + ((1.0 - ALPHA) * dipp if RT_IR else 0.0) + RN * rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n * dm[:, None] + PN * rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD * rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n - zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt + zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos - zi; ii = (v - vmin[:, None]) / step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen - 1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen - 2).clamp_min(0)[:, None])
        t = ii - i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c + 1)
        eg = gA * (1 - t) + gB * t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None] - eg) / gs[:, None]).abs(); dp = d ** GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5 * dp), zero)
        if LKF > 0:
            lk = (1.0 - LKF) * lk + LKF                  # 一様混合床: 外れGRで正解粒子が即死しない
        lk = lk.clamp_min(1e-300)
        avg = (w * lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate - dipp) / PHYS_SIG; lkp = torch.exp(-0.5 * dphys * dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        if USE_A:                                   # ★ row錨(毎行独立掛け, mult=温度補正, ramp=遅延起動)
            asig = (ancs[:, i][:, None] * AMUL).clamp_min(1e-6)
            da = (v - anc[:, i][:, None]) / asig    # v = pos - zi = 粒子のTVT
            da2 = da * da
            if RAMP > 0:
                tf = ((cur - md[:, 0]).clamp_min(0.0) / RAMP).clamp(0.0, 1.0)[:, None]
                da2 = da2 * tf                       # 温度でランプイン(近錨域は錨を弱く)
            lka = torch.where(da2 < 1200.0, torch.exp(-0.5 * da2), zero).clamp_min(1e-300)
        else:
            lka = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w * lk * lkp * lka, w * lkp * lka)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new / ws.clamp_min(1e-300), torch.full_like(w, 1.0 / N))
        ne = (w * w).sum(1); need = act & ((1.0 / ne) < (RESAMP * N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype) * (1.0 / N)
            u = u0 + arN[None, :] / N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N - 1)
            pos_rs = torch.gather(pos, 1, idx) + RP * rn(); rate_rs = torch.gather(rate, 1, idx) + RR * rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0 / N), w)
        pts[:, i] = (w * (pos - zi)).sum(1); pm = torch.where(act, cur, pm)
    return pts, log_lik

def _pad(inps, device, dtype=DTYPE):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    anc = torch.zeros((W, Tmax), dtype=dtype); ancs = torch.full((W, Tmax), 1e9, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn - 1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G - 1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
        anc[b, :Tn] = torch.from_numpy(x["anc"].astype("float32"))
        if Tn < Tmax: anc[b, Tn:] = anc[b, Tn - 1]
        ancs[b, :Tn] = torch.from_numpy(x["ancs"].astype("float32"))
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir), anc=to(anc), ancs=to(ancs))

def run_forward_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pts, ll = _pf_core(rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
                       pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
                       pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
                       rep(pad["anc"]), rep(pad["ancs"]), P, N, device, gen)
    pts = pts.view(W, S, -1).double().cpu().numpy(); ll = ll.view(W, S).cpu().numpy()
    outs = []
    for w in range(W):
        T = len(inps[w]["md"]); pr = pts[w, :, :T]
        llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)])
            lk = np.where(np.isfinite(llw), llw - mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6))
            s = ww.sum()
            ww = ww / s if s > 1e-300 else np.full(S, 1.0 / S)
        else:
            ww = np.full(S, 1.0 / S)
        outs.append((ww[:, None] * pr).sum(0))
    return outs


# ----------------------- 評価(対象well指定) -----------------------

In [ ]:
# ===== test側キャッシュ + 構造prior(錨セットがある場合のみ) =====
NEED_ANCHOR = any(bool(P.get("use_anchor", False)) for P in PARAM_SETS)
STRUCT_T = {}
if NEED_ANCHOR:
    _fz = next((p for p in ([str(Path("/kaggle/input")) and q for q in _glob.glob("/kaggle/input/**/struct_field_v38.npz", recursive=True)]
                            + [r"C:\Users\kosaka256\Documents\rogii_claude\v64\artifacts_v64\struct_field_v38.npz"]) if os.path.exists(p)), None)
    if _fz is None:
        print("★警告: struct_field_v38.npz が見つからない → 錨セットは錨OFFで実行")
    else:
        FZ = np.load(_fz)
        FLD = FZ["field"]; KAP = FZ["kappa"]; CEN = FZ["centers"]; SGC = FZ["sigs"]
        # ===== livefield: test既知帯(prefix)から dip観測点を抽出して場に追加 =====
        _TH = float(FZ["theta0"]); _MINPROJ = 0.3; _MIN_LAT = 150
        _pts = []
        _wi0 = int(FLD[:, 3].max()) + 1 if FLD.shape[1] > 3 else 10 ** 6
        for _k, _fp in enumerate(_glob.glob(str(Path(TESTD) / "*__horizontal_well.csv"))):
            try:
                _df = pd.read_csv(_fp, usecols=["X", "Y", "Z", "TVT_input"])
                _X = _df["X"].to_numpy(); _Y = _df["Y"].to_numpy(); _Z = _df["Z"].to_numpy()
                _ti = _df["TVT_input"].to_numpy()
                _fin = np.where(np.isfinite(_ti))[0]
                if len(_fin) < _MIN_LAT + 2: continue
                _s = int(_fin.max())
                # 既知帯の水平区間(横ステップ > 2|dz| が安定して成り立つ範囲)のみ使用
                _dx = np.diff(_X[:_s + 1]); _dy = np.diff(_Y[:_s + 1]); _dz = np.diff(_Z[:_s + 1])
                _lat = np.hypot(_dx, _dy) > 2.0 * np.abs(_dz)
                _idx = np.where(_lat)[0]
                if len(_idx) < _MIN_LAT: continue
                _j0 = int(_idx[0])
                _n = _s - _j0
                if _n < _MIN_LAT: continue
                _ndz = -np.diff(_Z)[_j0:_s]; _nn = len(_ndz)
                _U = np.cumsum(_ndz)
                _R0 = _ti[_j0 + 1:_s + 1] - _ti[_j0]
                _m = np.isfinite(_R0)
                if _m.sum() < _MIN_LAT: continue
                _Kp = int(np.clip(_nn // 200, 1, 8))
                _edges = np.linspace(0, _nn, _Kp + 1)
                _t = np.arange(1, _nn + 1.0)
                _phi = np.column_stack([np.clip(_t - _edges[_j], 0, _edges[_j + 1] - _edges[_j]) for _j in range(_Kp)])
                _c = np.linalg.lstsq(_phi[_m], (_R0 - _U)[_m], rcond=None)[0]
                for _j in range(_Kp):
                    _f0 = _j0 + 1 + int(_edges[_j]); _f1 = min(_j0 + 1 + max(int(_edges[_j + 1]) - 1, int(_edges[_j])), len(_X) - 1)
                    _az = np.degrees(np.arctan2(_Y[_f1] - _Y[_f0], _X[_f1] - _X[_f0])) % 360
                    _pj = np.cos(np.radians(_az - _TH))
                    if abs(_pj) > _MINPROJ and np.isfinite(_c[_j]):
                        _pts.append(((_X[_f0] + _X[_f1]) / 2, (_Y[_f0] + _Y[_f1]) / 2, float(_c[_j] / _pj), _wi0 + _k))
            except Exception:
                continue
        if _pts:
            _pts = np.array(_pts, np.float64)
            FLD = np.vstack([FLD[:, :4] if FLD.shape[1] >= 4 else np.column_stack([FLD, -np.ones(len(FLD))]), _pts])
            print(f"[livefield] train場 {len(FZ['field'])}点 + test既知帯 {len(_pts)}点"
                  f"({len(set(_pts[:, 3]))}坑井) = {len(FLD)}点")
        else:
            print("[livefield] ★test既知帯から観測点が取れず → train場のみ(通常struct20と同一)")
        TH = float(FZ["theta0"]); K = int(FZ["K"]); KK = int(FZ["knn_k"]); KH = float(FZ["knn_h"])
        GT = float(FZ["gate"]); KB = FZ["kbins"]; SMIN = float(FZ["sig_min"]); SMAX = float(FZ["sig_max"])
        for fp in _glob.glob(str(TESTD / "*__horizontal_well.csv")):
            wid = os.path.basename(fp).split("__")[0]
            try:
                df = pd.read_csv(fp, usecols=["X", "Y", "Z", "TVT_input"])
                X = df["X"].to_numpy(); Y = df["Y"].to_numpy(); Z = df["Z"].to_numpy(); ti = df["TVT_input"].to_numpy()
                fin = np.where(np.isfinite(ti))[0]
                if len(fin) == 0: continue
                s = int(fin.max()); ndz = -np.diff(Z)[s:]; n = len(ndz)
                if n < 2: continue
                edges = np.linspace(0, n, K + 1)
                segid = np.clip(np.searchsorted(edges[1:], np.arange(1, n + 1.0), side="left"), 0, K - 1)
                mid = np.empty((K, 2)); proj = np.empty(K)
                for j in range(K):
                    f0 = s + 1 + int(edges[j]); f1 = min(s + 1 + max(int(edges[j + 1]) - 1, int(edges[j])), len(X) - 1)
                    az = np.degrees(np.arctan2(Y[f1] - Y[f0], X[f1] - X[f0])) % 360
                    mid[j] = ((X[f0] + X[f1]) / 2, (Y[f0] + Y[f1]) / 2); proj[j] = np.cos(np.radians(az - TH))
                Dh = np.empty(K); dd = np.empty(K)
                for j in range(K):
                    d2 = (FLD[:, 0] - mid[j, 0]) ** 2 + (FLD[:, 1] - mid[j, 1]) ** 2
                    sel = np.argpartition(d2, KK)[:KK]
                    wv = np.exp(np.maximum(-d2[sel] / (2 * KH ** 2), -700))
                    Dh[j] = np.sum(wv * FLD[sel, 2]) / np.sum(wv); dd[j] = np.sqrt(np.median(d2[sel]))
                gg = ndz + (Dh * proj)[segid]; gg = np.where(np.abs(proj[segid]) < GT, 0.0, gg)
                kstep = KAP[np.digitize(dd, KB[1:-1])][segid]
                tvt = float(ti[s]) + np.cumsum(kstep * gg)
                sig = np.clip(np.interp(dd[segid], CEN, SGC), SMIN, SMAX)
                STRUCT_T[wid] = dict(tvt=tvt.astype(np.float64), sig=sig.astype(np.float64))
            except Exception:
                continue
        print(f"[struct] test実行時構築: {len(STRUCT_T)} wells")

# --- self/nbr 用: train近傍プール(nbrセットがある場合のみ構築。test近傍=train坑井, ≤1500ft, ≤3本, 2本未満=tw) ---
MAX_DIST = 1500.0; K_MAX = 3; NEED_REFS = 2
NEED_NBR = any("nbr" in str(P.get("tw_source", "tw")) for P in PARAM_SETS)
TRAIND = DATA / "train"
_TRC = None; _REFC = {}
def _train_centroids():
    global _TRC
    if _TRC is None:
        wids = sorted(set(os.path.basename(f).split("__")[0] for f in _glob.glob(str(TRAIND / "*__horizontal_well.csv"))))
        xy = []
        for w_ in wids:
            h_ = pd.read_csv(TRAIND / f"{w_}__horizontal_well.csv", usecols=["X", "Y"])
            xy.append((float(h_["X"].mean()), float(h_["Y"].mean())))
        _TRC = (wids, np.array(xy))
        print(f"[nbr] trainプール: {len(wids)} wells")
    return _TRC
def _ref_pairs(w_):
    if w_ not in _REFC:
        h_ = pd.read_csv(TRAIND / f"{w_}__horizontal_well.csv", usecols=["GR", "TVT_input"])
        m_ = h_["TVT_input"].notna().to_numpy() & np.isfinite(h_["GR"].to_numpy(float))
        _REFC[w_] = (h_["TVT_input"].to_numpy(float)[m_], h_["GR"].to_numpy(float)[m_])
    return _REFC[w_]
def _nbr_pairs_of(cx, cy):
    wids, xy = _train_centroids()
    d = np.hypot(xy[:, 0] - cx, xy[:, 1] - cy)
    order = np.argsort(d)
    refs = [wids[j] for j in order if d[j] <= MAX_DIST][:K_MAX]
    if len(refs) < NEED_REFS: return None
    ts = []; gs_ = []
    for r_ in refs:
        t_, g_ = _ref_pairs(r_); ts.append(t_); gs_.append(g_)
    return (np.concatenate(ts), np.concatenate(gs_))

def build_cache_test(wid):
    """v65 build_cache の test版(TVT不要)。返り値: make_inputs 互換 cache dict + 行index。"""
    hw = pd.read_csv(TESTD / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TESTD / f"{wid}__typewell.csv").sort_values("TVT")
    kn = hw["TVT_input"].notna(); ev = ~kn
    if kn.sum() < 10 or ev.sum() == 0: return None
    tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    fin = np.isfinite(tw_tvt) & np.isfinite(tw_gr); tw_tvt, tw_gr = tw_tvt[fin], tw_gr[fin]
    if len(tw_tvt) < 3: return None
    tw_fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.
    gr_filled = hw["GR"].astype(float).interpolate(limit_direction="both").fillna(tw_fb).to_numpy(float)
    ev_pos = hw.index.get_indexer(hw[ev].index); kpos = hw.index.get_indexer(hw[kn].index)
    kn_tvtin = hw["TVT_input"].to_numpy(float)[kpos]
    ls = float(kn_tvtin[-1] + hw["Z"].to_numpy(float)[kpos][-1])
    tl = hw[kn].tail(30)
    dt = np.diff(tl["TVT_input"].values); dz = np.diff(tl["Z"].values); dm = np.diff(tl["MD"].values)
    m = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.
    nev = int(ev.sum())
    sp = STRUCT_T.get(wid)
    if sp is not None and len(sp["tvt"]) == nev:
        anc = np.asarray(sp["tvt"], np.float64); ancs = np.asarray(sp["sig"], np.float64)
    else:
        anc = np.zeros(nev); ancs = np.full(nev, 1e9)
    nbr_pairs = _nbr_pairs_of(float(hw["X"].mean()), float(hw["Y"].mean())) if NEED_NBR else None
    return dict(md=hw["MD"].to_numpy(float)[ev_pos], z=hw["Z"].to_numpy(float)[ev_pos],
                gr_filled=gr_filled, ev_pos=ev_pos, kpos=kpos, kn_tvtin=kn_tvtin,
                tw_tvt=tw_tvt, tw_gr=tw_gr, tw_fb=tw_fb, ls=ls, ir=ir,
                anc=anc, ancs=ancs, ev_index=np.where(ev.to_numpy())[0], lk=float(kn_tvtin[-1]),
                nbr_pairs=nbr_pairs)

In [ ]:
# ===== 実行: 各セット → seed尤度加重平均 → セット間 AVG 合成 =====
t0 = time.time()
wids = sorted(set(os.path.basename(f).split("__")[0] for f in _glob.glob(str(TESTD / "*__horizontal_well.csv"))))
caches = {}
for w in wids:
    c = build_cache_test(w)
    if c is not None: caches[w] = c
W = sorted(caches.keys())
print(f"test wells: {len(W)}/{len(wids)}")

preds_by_set = []
for si, P0 in enumerate(PARAM_SETS):
    P = dict(P0)
    outs = {}
    for c0_ in range(0, len(W), CHUNK):
        sub = W[c0_:c0_ + CHUNK]
        inps = [make_inputs(caches[w], P) for w in sub]
        res = run_forward_ens(inps, P, DEVICE, SEED, N_SEEDS)
        for j, w in enumerate(sub): outs[w] = np.asarray(res[j], np.float64)
        if DEVICE == "cuda": torch.cuda.empty_cache()
    preds_by_set.append(outs)
    print(f"set{si+1}/{len(PARAM_SETS)} done ({time.time()-t0:.0f}s)", flush=True)

pids = []; pvals = []
for w in W:
    c = caches[w]
    A = np.stack([ps[w] for ps in preds_by_set], 0)            # (S, nev) 各セットのTVT軌跡
    z = np.nanmedian(A, 0) if AVG == "median" else np.nanmean(A, 0)
    if PP:
        mds = np.maximum(c["md"] - c["md"][0] + 1.0, 0.0)
        d = (z - c["lk"]) * (1.0 - np.exp(-mds / float(PP["tau"]))) * float(PP.get("alpha", 1.0))
        z = c["lk"] + d
    pids += [f"{w}_{int(i)}" for i in c["ev_index"]]
    pvals += list(z)
pred_df = pd.DataFrame({"id": pids, "tvt": pvals})
print("predicted rows:", len(pred_df), f"({time.time()-t0:.0f}s)")

In [ ]:
# ===== submission(欠け行は lk フォールバック) =====
ss = pd.read_csv(DATA / "sample_submission.csv")
sub = ss[["id"]].merge(pred_df, on="id", how="left")
miss = sub["tvt"].isna()
if miss.any():
    lkmap = {}
    for wid in set(i.rsplit("_", 1)[0] for i in sub.loc[miss, "id"]):
        h = pd.read_csv(TESTD / f"{wid}__horizontal_well.csv", usecols=["TVT_input"])
        v = h["TVT_input"].dropna()
        lkmap[wid] = float(v.iloc[-1]) if len(v) else 0.0
    sub.loc[miss, "tvt"] = [lkmap[i.rsplit("_", 1)[0]] for i in sub.loc[miss, "id"]]
    print(f"lkフォールバック: {int(miss.sum())} rows")
sub.to_csv("submission.csv", index=False)
print("submission.csv:", len(sub), "rows")